# scWAT Xenium evidence-only QC and downstream-input summary

This notebook is the sole reader-facing QC report for the four independently processed scWAT sections. Sections are technical units; the two mice are the biological units. Confirmed cycle-to-codeword mapping is unavailable, so decisions use only the frozen cross-section evidence rules.

## Setup

### Parameters

Inputs: four completed section bundles below `RUN_ROOT`, the verified sample manifest, and versioned QC settings. Outputs: combined tables and Cell-style figures below `${RUN_ROOT}/slide_summary/`.

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
RUN_LABEL <- "full_notebook_qc_v2"
EXPECTED_SECTION_COUNT <- 4L
METADATA_PATH <- file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv")
EXTENDED_QC_CONFIG_PATH <- file.path(PIPELINE_REPO, "config", "extended_qc_defaults.tsv")
SUBSET_REFERENCE_PATH <- file.path(PIPELINE_REPO, "config", "subset_qc_reference.tsv")


In [ ]:
RUN_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
assert_path_within(PROJECT_ROOT, RUN_ROOT)
assert_path_within(PROJECT_ROOT, tempdir())
stopifnot(EXPECTED_SECTION_COUNT == 4L)
cat("Slide QC run root:", RUN_ROOT, "\n")


## Inputs and validation

Exactly four independently completed core and extended section bundles are required. The verified manifest maps 62308/62309 to Mouse 1 and 62310/62311 to Mouse 2. Left/right is not an analysis factor.

In [ ]:
coverage <- validate_four_section_outputs(RUN_ROOT, paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
extended_coverage <- validate_four_extended_section_outputs(RUN_ROOT, coverage$region_id)
stopifnot(identical(coverage$region_id, extended_coverage$region_id))
slide_data <- read_slide_qc_outputs(RUN_ROOT, coverage$region_id)
slide_summary <- summarise_slide_qc(slide_data)
extended_slide_data <- read_extended_slide_qc_outputs(RUN_ROOT, coverage$region_id)
manifest <- utils::read.delim(METADATA_PATH, check.names = FALSE)
extended_config <- read_extended_qc_config(EXTENDED_QC_CONFIG_PATH)
subset_reference <- utils::read.delim(SUBSET_REFERENCE_PATH, check.names = FALSE)
extended_slide_summary <- summarise_extended_slide_qc(extended_slide_data, slide_summary$section_summary, manifest, extended_config, subset_reference)
eos_gene_sets <- utils::read.delim(file.path(PIPELINE_REPO, "config", "eos_gene_sets.tsv"), check.names = FALSE)
release_provenance <- paste(RUN_LABEL, unique(extended_coverage$mode), normalizePath(RUN_ROOT, winslash = "/", mustWork = TRUE), sep = "|")
evidence_summary <- summarise_evidence_only_qc(
  extended_slide_data, extended_slide_summary$candidates, eos_gene_sets,
  RUN_LABEL, unique(extended_coverage$mode), release_provenance
)
stopifnot(length(unique(slide_data$cell_metadata$region_id)) == 4L)
list(core = coverage, extended = extended_coverage, cells = nrow(slide_data$cell_metadata))


## TL;DR and QC decision

Fixed evidence-only status: Region 1 `PRIMARY_CONDITIONAL`, Region 2 `PRIMARY_CONDITIONAL`, Region 3 `PRIMARY`, and Region 4 `SENSITIVITY_ONLY`. Region 4 cannot enter cluster discovery or primary gene-level results; it will later map to the finalized Region 1-3 reference, with low-confidence assignments labelled `Uncertain`.

In [ ]:
direct_alarm_regions <- unique(extended_slide_data$cycle_alarm_evidence$region_id[extended_slide_data$cycle_alarm_evidence$evidence_status == "DIRECT_EVIDENCE"])
qc_decision <- merge(evidence_summary$sections, data.frame(region_id = paste0("Region_",1:4), direct_poor_cycle_alarm = paste0("Region_",1:4) %in% direct_alarm_regions), by = "region_id", sort = FALSE)
qc_decision <- qc_decision[match(paste0("Region_",1:4), qc_decision$region_id), ]
qc_decision[, c("region_id", "section_status", "input_cells", "primary_include_cells", "strict_include_cells", "hotspot_sensitivity_include_cells", "direct_poor_cycle_alarm", "cluster_discovery_eligible")]


## Core QC distributions

These are descriptive section-level and cell-level QC summaries. No cells are automatically deleted, and cell-level distributions do not create biological replication.

In [ ]:
slide_summary$section_summary
stopifnot(all(slide_summary$section_summary$cells_deleted == 0L))
slide_plots <- plot_slide_qc(slide_data, slide_summary)
for (plot in slide_plots) print(plot)


## Alarm evidence and evidence-only gene tiers

No gene is described as confirmed affected or confirmed unaffected. All 479 genes remain `RAW_COMPLETE_PANEL`; the full-data contract expects 67 `CONSERVATIVE_NO_SIGNAL_DETECTED`, 245 `PROVISIONAL_PRIMARY_FEATURES`, and 234 `TECHNICAL_RISK_SENSITIVITY_ONLY`. The conservative 67 are a subset of the provisional 245. The 234 risk genes cannot define primary clusters.

In [ ]:
extended_slide_data$cycle_alarm_evidence
gene_tier_counts <- data.frame(
  decision = c("RAW_COMPLETE_PANEL", "CONSERVATIVE_NO_SIGNAL_DETECTED", "PROVISIONAL_PRIMARY_FEATURES", "TECHNICAL_RISK_SENSITIVITY_ONLY"),
  genes = c(nrow(evidence_summary$genes), sum(evidence_summary$genes$conservative_evidence_status == "CONSERVATIVE_NO_SIGNAL_DETECTED"), sum(evidence_summary$genes$primary_feature_status == "PROVISIONAL_PRIMARY_FEATURES"), sum(evidence_summary$genes$technical_risk_status == "TECHNICAL_RISK_SENSITIVITY_ONLY"))
)
gene_tier_counts
with(evidence_summary$eos[evidence_summary$eos$retained_provisional, ], table(gene_set))


## Subset versus full-data burden

The comparison is descriptive across four technical sections. `NOT_RUN_LOCAL_SUBSET` means the full-data ranking remains an HPC checkpoint; rank correlations across only four sections are not biological evidence.

In [ ]:
extended_slide_summary$ranking
extended_slide_summary$rank_agreement


## Spatial QC

Global kNN clustering, tissue-edge proxies, dense-cell proxies, and candidate hotspot bins are coordinate-based diagnostics. A hotspot is only `MORPHOLOGY_REVIEW_REQUIRED`; folds, tears, tissue edges, and aggregates require image review.

In [ ]:
extended_slide_data$spatial_global
extended_slide_data$spatial_edge_density
evidence_summary$hotspots


## Within-mouse concordance

The verified technical pairs are 62308/62309 for Mouse 1 and 62310/62311 for Mouse 2. Thresholds are advisory. The Region 3/4 cell-area contrast is reviewed separately because it was not part of the original concordance gate.

In [ ]:
extended_slide_summary$concordance$summary


## Diagnostic Cell-style figures

Colors and scales are consistent across sections where scientifically appropriate. These plots support review rather than biological inference.

In [ ]:
extended_slide_plots <- plot_extended_slide_qc(extended_slide_data, extended_slide_summary)
for (plot in extended_slide_plots) print(plot)
mask_plot_data <- evidence_summary$sections[, c("region_id", "primary_include_cells", "strict_include_cells", "hotspot_sensitivity_include_cells")]
mask_long <- reshape(mask_plot_data, varying = names(mask_plot_data)[-1], v.names = "cells", timevar = "mask", times = names(mask_plot_data)[-1], direction = "long")
print(ggplot2::ggplot(mask_long, ggplot2::aes(region_id, cells, fill = mask)) + ggplot2::geom_col(position = "dodge") + ggplot2::scale_fill_manual(values = c(primary_include_cells="#3C5488", strict_include_cells="#00A087", hotspot_sensitivity_include_cells="#E64B35")) + ggplot2::labs(title="Downstream inclusion masks", x=NULL, y="Cells", fill=NULL) + cell_style_theme())
print(ggplot2::ggplot(gene_tier_counts, ggplot2::aes(reorder(decision, genes), genes, fill = decision)) + ggplot2::geom_col(show.legend=FALSE) + ggplot2::coord_flip() + ggplot2::labs(title="Evidence-only gene decisions", x=NULL, y="Genes") + cell_style_theme())


## Final QC decision and next actions

Phase 0-2 prepares immutable downstream inputs; it does not claim PCA, integration, clustering, Eos stability, or Region 4 mapping results. These checks therefore remain `PENDING_DOWNSTREAM_ANALYSIS`. Primary release stops automatically if any completed downstream gate becomes `STOP`.

In [ ]:
evidence_summary$release
cat("Evidence-only primary release:", evidence_summary$release$gate_status[evidence_summary$release$gate_id == "overall_primary_release"], "\n")


## Outputs and reload checks

Required outputs are `cell_downstream_masks.tsv.gz`, `section_downstream_decision.tsv`, `gene_downstream_decision.tsv`, `eos_gene_decision_summary.tsv`, `hotspot_sensitivity_decision.tsv`, and `evidence_only_qc_release.tsv`. Final raw-count bundles and `downstream_input_manifest.tsv` are written below `${RUN_ROOT}/downstream_inputs/`.

In [ ]:
slide_artifacts <- write_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, slide_data, slide_summary, slide_plots)
extended_slide_artifacts <- write_extended_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, extended_slide_data, extended_slide_summary, extended_slide_plots)
saved_summary <- readRDS(file.path(RUN_ROOT, "slide_summary", "slide_qc_summary.rds"))
stopifnot(nrow(saved_summary$data$coverage) == 4L)
stopifnot(length(unique(saved_summary$data$cell_metadata$region_id)) == 4L)
stopifnot(validate_extended_slide_qc_artifacts(RUN_ROOT, stop_on_error = TRUE))
evidence_only_artifacts <- write_evidence_only_qc_artifacts(PROJECT_ROOT, RUN_ROOT, evidence_summary)
stopifnot(validate_evidence_only_qc_artifacts(RUN_ROOT, stop_on_error = TRUE))
list(core = data.frame(artifact = basename(slide_artifacts), path = slide_artifacts),
     extended = data.frame(artifact = basename(extended_slide_artifacts), path = extended_slide_artifacts),
     evidence_only = data.frame(artifact = basename(evidence_only_artifacts), path = evidence_only_artifacts))
